# 01 – Data Audit & Label Validation

This notebook verifies the output of `python -m aml_benchmark.data.make_dataset`.

It checks:
1. Row counts and date range
2. Class distribution
3. Label consistency (patterns vs CSV label)
4. Basic value distributions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Make sure the package is importable when running the notebook directly
sys.path.insert(0, str(Path('..') / 'src'))

from aml_benchmark.config import PathConfig
from aml_benchmark.utils.io import load_parquet

paths = PathConfig()
print('Project root:', paths.raw_dir.parent)

In [ ]:
df = load_parquet(paths.output_transactions_labeled)
print(f'Shape: {df.shape}')
df.head(3)

## 1 · Schema and dtypes

In [ ]:
df.dtypes

In [ ]:
print('Timestamp range:')
print(f'  min: {df["timestamp"].min()}')
print(f'  max: {df["timestamp"].max()}')
print(f'  span: {df["timestamp"].max() - df["timestamp"].min()}')

## 2 · Class distribution

In [ ]:
vc = df['label'].value_counts()
print(vc)
print(f'\nIllicit ratio: {vc[1] / len(df):.4%}')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
vc.plot.bar(ax=ax, color=['steelblue', 'firebrick'])
ax.set_xticklabels(['Normal (0)', 'Illicit (1)'], rotation=0)
ax.set_ylabel('Count')
ax.set_title('Label distribution')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 3 · Label consistency audit

In [ ]:
print('label_from_patterns  vs  label_existing_csv cross-tab:')
print(pd.crosstab(df['label_from_patterns'], df['label_existing_csv'],
                  rownames=['pattern'], colnames=['csv']))
print(f'\nMismatch rows: {df["mismatch_flag"].sum():,}')

In [ ]:
if df['mismatch_flag'].sum() > 0:
    print('Sample mismatch rows:')
    display(df[df['mismatch_flag'] == 1].head(10))

## 4 · Feature value checks

In [ ]:
print('Payment format distribution:')
print(df['payment_format'].value_counts())

In [ ]:
print('Payment currency distribution:')
print(df['payment_currency'].value_counts())

In [ ]:
print('amount_paid summary:')
print(df['amount_paid'].describe())

## 5 · Temporal distribution

In [ ]:
daily = df.set_index('timestamp').resample('D')['label'].agg(['count', 'sum'])
daily.columns = ['total', 'illicit']
daily['illicit_ratio'] = daily['illicit'] / daily['total']

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
daily['total'].plot(ax=axes[0], color='steelblue', label='All transactions')
daily['illicit'].plot(ax=axes[0], color='firebrick', label='Illicit')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_title('Daily transaction volume')

daily['illicit_ratio'].plot(ax=axes[1], color='darkorange')
axes[1].set_ylabel('Illicit ratio')
axes[1].set_title('Daily illicit ratio')

plt.tight_layout()
plt.show()

## 6 · Quick null check

In [ ]:
null_counts = df.isnull().sum()
print('Null counts per column:')
print(null_counts[null_counts > 0] if null_counts.any() else 'None – clean!')